In [1]:
from urllib.request import Request, urlopen
from bs4 import BeautifulSoup

email = "skkufi0113@skkukdp.re.kr"

document_url = (
    "https://www.sec.gov/Archives/edgar/data/"
    "789019/000119312526323660/msft-20260630.htm"
)

request = Request(
    document_url,
    headers={"User-Agent": f"Twodigit Research {email}"},
)

with urlopen(request, timeout=30) as response:
    status = response.status
    html_bytes = response.read()

soup = BeautifulSoup(html_bytes, "html.parser")

print("HTTP 상태:", status)
print("받은 크기:", len(html_bytes), "바이트")
print("HTML 표 요소 수:", len(soup.find_all("table")))

HTTP 상태: 200
받은 크기: 8585619 바이트
HTML 표 요소 수: 88


In [2]:
# 1. 회사 사업 설명이 포함된 문단 찾기
paragraphs = soup.find_all("p")

business_candidates = []

for index, paragraph in enumerate(paragraphs):
    text = paragraph.get_text(" ", strip=True)

    if len(text) >= 150 and "Microsoft" in text:
        business_candidates.append((index, text))

print("[본문 후보: 최대 3개]")
for index, text in business_candidates[:3]:
    print(f"\n문단 인덱스: {index}")
    print(text[:1200])

# 2. Revenue와 Net income이 함께 있는 표 찾기
tables = soup.find_all("table")
candidate_indices = []

print("\n[손익계산서 후보 표]")

for index, table in enumerate(tables):
    text = table.get_text(" ", strip=True)

    if "revenue" in text.lower() and "net income" in text.lower():
        candidate_indices.append(index)
        print(f"\n표 인덱스: {index}")
        print(text[:2000])

print("\n후보 표 인덱스:", candidate_indices)

[본문 후보: 최대 3개]

문단 인덱스: 526
Microsoft is a technology company committed to making digital technology and artificial intelligence (“AI”) available broadly and doing so responsibly. Our mission is to empower every person and every organization on the planet to achieve more.

문단 인덱스: 528
AI is fundamentally transforming productivity for every individual, organization, and industry. Microsoft's AI offerings span every layer of the technology stack, enabling transformative outcomes across sectors and unlocking opportunity for every country, community, and individual.

문단 인덱스: 539
The Microsoft Cloud provides integration across the technology stack while offering openness, improving time to value, reducing costs, and increasing agility. Our cloud business benefits from three economies of scale: datacenters that deploy computational resources at significantly lower cost per unit than smaller ones; datacenters that coordinate and aggregate diverse customer, geographic, and application demand p

In [3]:
from pathlib import Path
import json

output_dir = Path.cwd()

# D15 폴더에서만 저장
if output_dir.name != "D15":
    raise ValueError(f"D15 폴더가 아닙니다: {output_dir}")

paragraph_index = 526
table_index = 24

paragraph = paragraphs[paragraph_index]
table = tables[table_index]

body_text = paragraph.get_text(" ", strip=True)
table_text = table.get_text(" ", strip=True)

# 선택한 내용이 예상한 본문·표인지 확인
if "Microsoft is a technology company" not in body_text:
    raise ValueError("본문 내용이 달라졌습니다. 후보를 다시 확인하세요.")

if not all(
    word in table_text
    for word in ["Total revenue", "Operating income", "Net income", "2026"]
):
    raise ValueError("선택한 표를 다시 확인하세요.")

# 본문과 표를 각각 저장
body_path = output_dir / "D15_MSFT_body_sample.txt"
table_path = output_dir / "D15_MSFT_income_statement.html"
source_path = output_dir / "D15_MSFT_source_record.json"

body_path.write_text(body_text, encoding="utf-8")

table_html = (
    '<!doctype html><html><head><meta charset="utf-8">'
    '<title>MSFT income statement sample</title>'
    '</head><body>'
    + str(table)
    + '</body></html>'
)
table_path.write_text(table_html, encoding="utf-8")

# 어디서 추출했는지 기록
source_record = {
    "source_url": document_url,
    "cik": "0000789019",
    "accession_number": "0001193125-26-323660",
    "document": "msft-20260630.htm",
    "form": "10-K",
    "report_date": "2026-06-30",
    "paragraph_index_zero_based": paragraph_index,
    "table_index_zero_based": table_index,
    "body_search_phrase": "Microsoft is a technology company",
    "table_search_phrase": "Total revenue",
    "table_years": [2026, 2025, 2024],
    "unit_note": "In millions, except per share amounts",
    "body_file": body_path.name,
    "table_file": table_path.name,
}

source_path.write_text(
    json.dumps(source_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("본문 저장:", body_path)
print("표 저장:", table_path)
print("출처 기록:", source_path)
print("본문 저장 일치:",
      body_path.read_text(encoding="utf-8") == body_text)
print("표 저장 일치:",
      table_path.read_text(encoding="utf-8") == table_html)

본문 저장: C:\ai_campus_main_proj\data_research\D15\D15_MSFT_body_sample.txt
표 저장: C:\ai_campus_main_proj\data_research\D15\D15_MSFT_income_statement.html
출처 기록: C:\ai_campus_main_proj\data_research\D15\D15_MSFT_source_record.json
본문 저장 일치: True
표 저장 일치: True
